# Phân tích xu hướng mạng xã hội tiếng Việt bằng Topic Modeling (LDA)
Notebook này triển khai prototype cho việc khám phá chủ đề (Topic Modeling) từ các văn bản mạng xã hội tiếng Việt bằng mô hình Latent Dirichlet Allocation (LDA).

In [1]:
# Cài đặt các thư viện cần thiết (nếu chưa có)
# !pip install gensim pyldavis underthesea pandas

import os
import re
import json
import logging
import warnings
from pprint import pprint
import pandas as pd

# Data manipulation
import pandas as pd

# NLP & Topic Modeling
from underthesea import word_tokenize
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore

# Visualization
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

## 1. Data Loading
Đọc danh sách stopwords và từ điển teencode/slang. Sử dụng biến `DATA_DIR` để trỏ tới thư mục `data/`.

In [2]:
# Cấu hình đường dẫn
DATA_DIR = "../data"
STOPWORDS_FILE = os.path.join(DATA_DIR, "stopwords_vi.txt")
SLANG_FILE = os.path.join(DATA_DIR, "slang_dict.json")
FAKE_DIR     = os.path.join(DATA_DIR, "fake")
POSTS_CSV    = os.path.join(FAKE_DIR, "posts_5k.csv")
COMMENTS_CSV = os.path.join(FAKE_DIR, "comments_5k.csv")


def load_csv_texts(csv_path: str, text_cols: list) -> list:
    """
    Đọc file CSV và ghép các cột text thành danh sách chuỗi.
    """
    try:
        df = pd.read_csv(csv_path, encoding="utf-8")
        fname = os.path.basename(csv_path)
        print(f"  📄 {fname}: {len(df):,} hàng | cột: {df.columns.tolist()}")
        existing = [c for c in text_cols if c in df.columns]
        if not existing:
            print(f"  ⚠️  Không tìm thấy cột {text_cols} — bỏ qua.")
            return []
        texts = (
            df[existing]
            .fillna("")
            .apply(lambda row: " ".join(str(v) for v in row if str(v).strip()), axis=1)
            .tolist()
        )
        return texts
    except FileNotFoundError:
        print(f"❌ File không tồn tại: {csv_path}")
        return []
    except Exception as e:
        print(f"❌ Lỗi đọc CSV: {e}")
        return []
# ── Load & gộp corpus ────────────────────────────────────────
print("📂 Đang đọc dữ liệu từ data/fake/ ...")
post_texts    = load_csv_texts(POSTS_CSV,    text_cols=["tieu_de", "noi_dung"])
comment_texts = load_csv_texts(COMMENTS_CSV, text_cols=["comment"])
documents = post_texts + comment_texts
print(f"\n✅ Tổng số văn bản: {len(documents):,}")
print("\n--- Ví dụ 3 văn bản đầu ---")
for i, doc in enumerate(documents[:3]):
    print(f"[{i+1}] {doc[:130]}...")

def load_stopwords(filepath):
    """Đọc file stopwords, mỗi dòng một từ"""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            stopwords = set([line.strip() for line in f if line.strip()])
        print(f"✅ Đã tải {len(stopwords)} stopwords.")
        return stopwords
    except FileNotFoundError:
        print(f"❌ Không tìm thấy file stopwords tại {filepath}. Sẽ sử dụng set rỗng.")
        return set()
    except Exception as e:
        print(f"❌ Lỗi khi đọc file stopwords: {e}")
        return set()

def load_slang_dict(filepath):
    """Đọc dictionary chuẩn hóa teencode ở định dạng JSON"""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            slang_dict = json.load(f)
        print(f"✅ Đã tải {len(slang_dict)} từ lóng/teencode.")
        return slang_dict
    except FileNotFoundError:
        print(f"❌ Không tìm thấy file slang dictionary tại {filepath}. Sẽ sử dụng dict rỗng.")
        return {}
    except json.JSONDecodeError:
        print(f"❌ File slang dictionary không đúng định dạng JSON.")
        return {}
    except Exception as e:
        print(f"❌ Lỗi khi đọc file slang dictionary: {e}")
        return {}

# Load dữ liệu
stopwords_vi = load_stopwords(STOPWORDS_FILE)
slang_dict = load_slang_dict(SLANG_FILE)

📂 Đang đọc dữ liệu từ data/fake/ ...
  📄 posts_5k.csv: 1,050 hàng | cột: ['id', 'tac_gia', 'tieu_de', 'thoi_gian', 'noi_dung']
  📄 comments_5k.csv: 4,186 hàng | cột: ['id_comment', 'user', 'time', 'comment', 'id_post']

✅ Tổng số văn bản: 5,236

--- Ví dụ 3 văn bản đầu ---
[1] So sánh Kia EV VF 9 vs BYD bZ4X: Nên mua xe điện nào 1200tr? vả lại chia sẻ chi phí thực tế 2 tháng dùng Tesla VF 9. Uh huh, điện:...
[2] Samsung X100 Ultra giảm còn 32tr: Chốt ngay hay chờ thêm? Mình dùng OnePlus 9 Pro thay mirrorless 9 tháng r. Kết quả: 80% tình huố...
[3] Thực đơn Highlands có gì ngon? Review toàn bộ menu giá 25k 💪💯 Nha mn, thế là chia sẻ góc nhìn về Gong Cha sau nhiều lần ghé. Menu:...
✅ Đã tải 1942 stopwords.
✅ Đã tải 2124 từ lóng/teencode.


Giả lập dữ liệu văn bản mạng xã hội (5000 văn bản). Do ở mức prototype, ta dùng một danh sách lặp lại để mô phỏng. Nếu dùng thực tế có thể đọc từ CSV bằng `pd.read_csv()`.

## 2. Text Preprocessing
Xây dựng một pipeline làm sạch văn bản nhằm chuẩn hóa dữ liệu trước khi đưa vào mô hình học máy:
1. Chuyển thành chữ thường.
2. Dùng `slang_dict.json` để chuẩn hóa teencode/từ viết tắt.
3. Loại bỏ dấu câu và ký tự đặc biệt.
4. Tokenize văn bản.
5. Loại bỏ stopwords.

In [3]:
def preprocess_text(text, stopwords, slang_dict):
    """
    Tiền xử lý văn bản tiếng Việt
    """
    if not isinstance(text, str):
        return []
    
    # 1. Chuyển thành chữ thường
    text = text.lower()
    
    # 2. Xóa các ký tự không phải chữ cái và số (giữ lại khoảng trắng)
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # 3. Chuẩn hóa teencode / từ viết tắt (slang)
    words = text.split()
    normalized_words = [slang_dict.get(word, word) for word in words]
    text = " ".join(normalized_words)
    
    # 4. Tokenize dùng underthesea (ghép từ bằng dấu '_')
    try:
        # format="text" sẽ trả về chuỗi với các từ ghép cách nhau bằng '_'
        tokens = word_tokenize(text, format="text").split()
    except Exception as e:
        # Fallback split nếu underthesea chưa import đúng hoặc gặp lỗi
        tokens = text.split()
        
    # 5. Loại bỏ stopwords và các token chỉ chứa số hoặc quá ngắn (chiều dài < 2, trừ một số chữ đặc biệt nếu cần)
    cleaned_tokens = [
        token for token in tokens
        if token not in stopwords and not token.isdigit() and len(token) >= 2
    ]
    
    return cleaned_tokens

# Chạy tiền xử lý trên toàn bộ corpus
print("Đang tiền xử lý dữ liệu...")
processed_docs = [preprocess_text(doc, stopwords_vi, slang_dict) for doc in documents]

# Hiển thị kết quả vài mẫu đầu tiên
print("\nKết quả tiền xử lý (5 văn bản đầu):")
for doc, p_doc in zip(documents[:5], processed_docs[:5]):
    print(f"Gốc: {doc}")
    print(f"Xử lý: {p_doc}\n")

Đang tiền xử lý dữ liệu...

Kết quả tiền xử lý (5 văn bản đầu):
Gốc: So sánh Kia EV VF 9 vs BYD bZ4X: Nên mua xe điện nào 1200tr? vả lại chia sẻ chi phí thực tế 2 tháng dùng Tesla VF 9. Uh huh, điện: trung bình 19.7 kWh/100km, tương đương ~690đ/km vs xăng ~1991đ/km. Tiết kiệm ~14.9tr/năm nếu chạy 13611km. Khum, bảo dưỡng: chưa mất gì ngoài kiểm tra định kỳ. Lốp mòn nhanh hơn xe xăng 1 chút do trọng lượng pin. autopilot giúp tiết kiệm thêm ~10% điện. Tổng thể xe điện kinh tế hơn hẳn xăng sau 2-3 năm dùng. 😍😎
Xử lý: ['so_sánh', 'kia', 'ev', 'vf', 'byd', 'bz4x', 'mua', 'xe_điện', 'tr_vả_lại', 'chia_sẻ', 'chi_phí', 'thực_tế', 'tesla', 'vf', 'huh', 'điện', 'trung_bình', 'kwh', 'km', 'tương_đương', 'km', 'xăng', 'km', 'tiết_kiệm', 'tr', 'chạy', 'km', 'bảo_dưỡng', 'kiểm_tra', 'định_kỳ_lốp', 'mòn', 'xe', 'xăng', 'chút', 'trọng_lượng', 'ghim', 'viết', 'autopilot', 'giúp', 'tiết_kiệm', 'điện', 'tổng_thể', 'xe_điện', 'kinh_tế', 'hẳn', 'xăng']

Gốc: Samsung X100 Ultra giảm còn 32tr: Chốt ngay hay 

## 3. Gensim Dictionary & Corpus
Tạo bộ từ điển (Dictionary) và biểu diễn các văn bản thành Bag-of-Words corpus sử dụng `gensim.corpora.Dictionary`.

In [4]:
# Tạo từ điển từ các văn bản đã tiền xử lý
dictionary = Dictionary(processed_docs)

# Lọc các từ xuất hiện quá ít (dưới 2 văn bản) hoặc quá nhiều (trên 50% số văn bản)
dictionary.filter_extremes(no_below=2, no_above=0.5)

# Tạo corpus: Biểu diễn văn bản ở dạng tần suất từ (Bag-of-Words)
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

print(f"Số lượng từ unique trong từ điển: {len(dictionary)}")
print(f"Số lượng véc-tơ văn bản trong corpus: {len(corpus)}")

Số lượng từ unique trong từ điển: 1037
Số lượng véc-tơ văn bản trong corpus: 5236


## 4. LDA Modeling
Huấn luyện mô hình LDA với số lượng topic tự chọn (ví dụ: `num_topics = 5`). Sử dụng `LdaMulticore` để tối ưu thời gian huấn luyện đa luồng.

In [7]:
# Cấu hình số lượng topics
num_topics = 10

print(f"Đang huấn luyện mô hình LDA với k={num_topics} ...")
lda_model = LdaMulticore(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    workers=4,        # Số luồng chạy (có thể tinh chỉnh tùy CPU)
    passes=10,        # Số vòng lặp qua corpus
    alpha='symmetric',
    random_state=42
)

# Hiển thị các từ khóa đại diện cho mỗi topic
print("\nTop các từ khóa cho mỗi topic:")
pprint(lda_model.print_topics())

Đang huấn luyện mô hình LDA với k=10 ...

Top các từ khóa cho mỗi topic:
[(0,
  '0.046*"giá" + 0.040*"app" + 0.039*"nhu_cầu" + 0.039*"xài" + 0.038*"tùy" + '
  '0.038*"kinh_nghiệm" + 0.037*"hardware" + 0.022*"xiaomi" + 0.015*"tính_năng" '
  '+ 0.012*"tr"'),
 (1,
  '0.030*"chuyên_nghiệp" + 0.025*"giá" + 0.023*"chụp" + 0.023*"camera" + '
  '0.022*"xiaomi" + 0.022*"oneplus" + 0.021*"ultra" + 0.019*"pixel" + '
  '0.019*"google" + 0.018*"samsung"'),
 (2,
  '0.040*"update" + 0.029*"version" + 0.024*"mấy" + 0.023*"lỗi" + '
  '0.021*"con_lăn" + 0.018*"viết" + 0.018*"cải_thiện" + 0.017*"tiếng" + '
  '0.015*"đồng_ý" + 0.015*"app"'),
 (3,
  '0.070*"bảo_hành" + 0.057*"lỗi" + 0.055*"ổn" + 0.037*"fix" + 0.020*"lắm" + '
  '0.020*"hơi" + 0.019*"ban_đầu" + 0.019*"kỳ_vọng" + 0.018*"vấn_đề" + '
  '0.018*"bao_lâu"'),
 (4,
  '0.048*"giá" + 0.033*"tr" + 0.030*"thực_sự" + 0.023*"mua" + 0.021*"chê" + '
  '0.020*"quảng_cáo" + 0.020*"uy_tín" + 0.020*"chỗ" + 0.020*"hcm" + '
  '0.019*"thử"'),
 (5,
  '0.037*"km" + 

## 5. Visualization
Dùng thư viện `pyLDAvis` để hiển thị trực quan biểu đồ liên kết giữa các topics và phân bố từ vựng.

In [8]:
# Cấu hình cài đặt pyLDAvis trong môi trường Notebook
pyLDAvis.enable_notebook()

# Chuẩn bị dữ liệu hiển thị (sử dụng pyLDAvis.gensim_models)
print("Đang chuẩn bị biểu đồ trực quan...")
vis_data = gensimvis.prepare(lda_model, corpus, dictionary)

# Hiển thị biểu đồ
pyLDAvis.display(vis_data)

Đang chuẩn bị biểu đồ trực quan...


## Kết luận
Notebook trên đã minh họa cơ bản quy trình chuẩn bị dữ liệu, làm sạch bằng slang dict/stopwords, huấn luyện mô hình Latent Dirichlet Allocation (LDA) để tìm ra Topic từ đoạn văn bản và trực quan hóa kết quả. Có thể phát triển chạy dữ liệu CSV có hàng triệu dòng khi thay đổi bước load data.